In [0]:
# =========================================================
# Pergunta: Times com maior posse de bola vencem mais?
# =========================================================

query = """
WITH resultado_time AS (
    SELECT
        p.partida_id,
        t.clube,
        CASE
            WHEN t.clube = p.mandante AND p.mandante_placar > p.visitante_placar THEN 'W'
            WHEN t.clube = p.visitante AND p.visitante_placar > p.mandante_placar THEN 'W'
            WHEN p.mandante_placar = p.visitante_placar THEN 'D'
            ELSE 'L'
        END AS resultado,
        t.posse_de_bola_num
    FROM fato_estatistica_time_partida t
    JOIN fato_partida p
        ON t.partida_id = p.partida_id
    WHERE t.posse_de_bola_num IS NOT NULL
),

quartis AS (
    SELECT
        posse_de_bola_num,
        NTILE(4) OVER (ORDER BY posse_de_bola_num) AS quartil,
        resultado
    FROM resultado_time
)

SELECT
    quartil,
    COUNT(*) AS jogos,
    SUM(CASE WHEN resultado = 'W' THEN 1 ELSE 0 END) AS vitorias,
    ROUND(
        CAST(SUM(CASE WHEN resultado = 'W' THEN 1 ELSE 0 END) AS DOUBLE)
        / CAST(COUNT(*) AS DOUBLE),
        3
    ) AS taxa_vitoria
FROM quartis
GROUP BY quartil
ORDER BY quartil
"""
# Executa a consulta SQL
df_resultado = spark.sql(query)

# Exibe o resultado no notebook
display(df_resultado)

quartil,jogos,vitorias,taxa_vitoria
1,1705,723,0.424
2,1705,666,0.391
3,1705,595,0.349
4,1705,509,0.299
